<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason%20Market%20Engine%20v1.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""
Jason Market Engine v1

Formål
------
Samler Market Solver + 5 GW Market Projection i ett ferdig motor-grensesnitt.

Dette scriptet er fortsatt isolert fra hoved-Jason. Det endrer ikke TO, Market_xP
eller PS-logikk. Det skriver kun rene motor-/kontrollark som Jason senere kan lese.

Kjørerekkefølge foreløpig:
1. jason_market_master.py
2. jason_market_solver_test_v1.py
3. jason_market_engine_v1.py

Viktig arkitektur
-----------------
Market_Engine_Wide_v1 og Market_Engine_Long_v1 skal være sannhetskilden resten av
Jason bruker for markedets 5 GW-lagprojeksjoner.

Datakilder
----------
- Market_Solver_Ratings_v1       : Solver Attack/DefWeak-ratings
- Market_Solver_Summary_v1       : mu, gamma, solver-feil og status
- Market_xG_All_Fixtures_v1      : ekte marked for neste GW
- FDR_Overview                   : fixture-matrise GW1-GW5 for projeksjon

Modell
------
Team xG = mu * Attack(team) * DefWeak(opponent) * H/A factor
H/A factor = gamma hvis laget er hjemme, 1/gamma hvis laget er borte.
CS% = exp(-Opp xG) * 100

Source-felt
-----------
- MARKET: ekte markedstall hentet fra Market_xG_All_Fixtures_v1
- PROJECTED: beregnet med Solver-ratinger og FDR_Overview
- PROJECTED_FDR_GW1_FALLBACK: brukes kun hvis ekte GW1-marked mangler

Sesongovergang
--------------
Hvis FDR_Overview fortsatt inneholder gamle lag, havner disse i Missing-arket.
Scriptet stopper ikke av den grunn.
"""

import math
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd

from google.colab import auth
import gspread
from google.auth import default

# -----------------------
# KONFIGURASJON
# -----------------------

SPREADSHEET_NAME = "Jason Development"

FDR_SHEET = "FDR_Overview"
SOLVER_RATINGS_SHEET = "Market_Solver_Ratings_v1"
SOLVER_SUMMARY_SHEET = "Market_Solver_Summary_v1"
MARKET_XG_SHEET = "Market_xG_All_Fixtures_v1"

ENGINE_SUMMARY_OUT = "Market_Engine_Summary_v1"
ENGINE_DASHBOARD_OUT = "Market_Engine_Dashboard_v1"
ENGINE_RATINGS_OUT = "Market_Engine_Ratings_v1"
ENGINE_LONG_OUT = "Market_Engine_Long_v1"
ENGINE_WIDE_OUT = "Market_Engine_Wide_v1"
ENGINE_MISSING_OUT = "Market_Engine_Missing_v1"

GW_COLUMNS = ["GW1", "GW2", "GW3", "GW4", "GW5"]
REAL_MARKET_GW = "GW1"

# FPL/FDR-kode -> vanlig markedsnavn.
CODE_TO_TEAM = {
    "ARS": "Arsenal",
    "AVL": "Aston Villa",
    "BHA": "Brighton and Hove Albion",
    "BOU": "Bournemouth",
    "BRE": "Brentford",
    "BUR": "Burnley",
    "CHE": "Chelsea",
    "CRY": "Crystal Palace",
    "EVE": "Everton",
    "FUL": "Fulham",
    "LEE": "Leeds United",
    "LIV": "Liverpool",
    "MCI": "Manchester City",
    "MUN": "Manchester United",
    "NEW": "Newcastle United",
    "NFO": "Nottingham Forest",
    "SUN": "Sunderland",
    "TOT": "Tottenham Hotspur",
    "WHU": "West Ham United",
    "WOL": "Wolverhampton Wanderers",
    "IPS": "Ipswich Town",
    "COV": "Coventry City",
    "HUL": "Hull City",
}

# -----------------------
# Google Sheets helpers
# -----------------------

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


def parse_float(v, default=np.nan):
    if v is None:
        return default
    s = str(v).strip()
    if s == "":
        return default
    s = s.replace(" ", "")
    if "," in s and "." not in s:
        s = s.replace(",", ".")
    elif "," in s and "." in s:
        s = s.replace(",", "")
    try:
        return float(s)
    except Exception:
        return default


def read_sheet(sheet_name):
    ws = sh.worksheet(sheet_name)
    values = ws.get_all_values()
    if not values:
        return pd.DataFrame()
    header = values[0]
    rows = values[1:]
    return pd.DataFrame(rows, columns=header)


def update_worksheet(sheet_name, dataframe, min_rows=100, min_cols=20, sleep_seconds=1.2):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).fillna("")
    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows=str(min_rows), cols=str(min_cols))

    ws.resize(
        rows=max(min_rows, len(df_clean) + 20),
        cols=max(min_cols, len(df_clean.columns) + 5),
    )
    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(sleep_seconds)


def norm_name(s):
    s = str(s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


def round_numeric(df, decimals=4):
    if df is None or df.empty:
        return df
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].round(decimals)
    return out

# -----------------------
# Les Solver-summary
# -----------------------

summary_df = read_sheet(SOLVER_SUMMARY_SHEET)
if summary_df.empty or not {"Metric", "Value"}.issubset(summary_df.columns):
    raise ValueError(f"{SOLVER_SUMMARY_SHEET} mangler Metric/Value")

summary_map = {str(r["Metric"]).strip(): r["Value"] for _, r in summary_df.iterrows()}
mu = parse_float(summary_map.get("mu / league avg xG"))
gamma = parse_float(summary_map.get("Gamma"))

if pd.isna(mu) or pd.isna(gamma):
    raise ValueError(f"Kunne ikke lese mu/gamma fra {SOLVER_SUMMARY_SHEET}")

solver_success = str(summary_map.get("Solver success", "")).strip()
solver_xg_mae = parse_float(summary_map.get("xG MAE"))
solver_xg_rmse = parse_float(summary_map.get("xG RMSE"))
solver_cs_mae = parse_float(summary_map.get("CS MAE pct points"))
regularization = parse_float(summary_map.get("Regularization"))
anchor_strength = parse_float(summary_map.get("Anchor strength"))

print(f"mu: {mu:.4f}")
print(f"gamma: {gamma:.4f}")
print(f"solver success: {solver_success}")

# -----------------------
# Les Solver-ratings
# -----------------------

ratings_df_raw = read_sheet(SOLVER_RATINGS_SHEET)
required_rating_cols = ["Team", "Solver Attack", "Solver Def Weakness"]
missing_rating_cols = [c for c in required_rating_cols if c not in ratings_df_raw.columns]
if missing_rating_cols:
    raise ValueError(f"{SOLVER_RATINGS_SHEET} mangler kolonner: {missing_rating_cols}")

ratings_df = ratings_df_raw.copy()
ratings_df["Team"] = ratings_df["Team"].astype(str).str.strip()
for col in ratings_df.columns:
    if col != "Team":
        ratings_df[col] = ratings_df[col].apply(parse_float)
ratings_df = ratings_df.dropna(subset=["Solver Attack", "Solver Def Weakness"]).copy()

name_lookup = {}
for _, r in ratings_df.iterrows():
    name_lookup[norm_name(r["Team"])] = r["Team"]


def resolve_team_name(name):
    """Returnerer Solver-teamnavn fra et markedsnavn, eller None."""
    n_cand = norm_name(name)
    if not n_cand:
        return None
    if n_cand in name_lookup:
        return name_lookup[n_cand]
    for n, team in name_lookup.items():
        if n.startswith(n_cand) or n_cand.startswith(n):
            return team
    loose_pairs = [
        ("brightonandhovealbion", ["brightonandhove", "brighton"]),
        ("wolverhamptonwanderers", ["wolverhampton", "wolves"]),
        ("tottenhamhotspur", ["tottenham"]),
        ("newcastleunited", ["newcastle"]),
        ("manchesterunited", ["manchesterunited", "manunited"]),
        ("nottinghamforest", ["nottinghamforest", "nottinghamfore"]),
        ("westhamunited", ["westham"]),
    ]
    for canonical, alts in loose_pairs:
        if n_cand == canonical or n_cand in alts:
            for alt in alts + [canonical]:
                for n, team in name_lookup.items():
                    if n.startswith(alt) or alt.startswith(n):
                        return team
    return None


def resolve_team_from_code(code):
    code = str(code or "").strip().upper()
    if not code:
        return None
    candidate = CODE_TO_TEAM.get(code)
    if not candidate:
        return None
    return resolve_team_name(candidate)


def code_from_team_name(team_name):
    resolved = resolve_team_name(team_name)
    if not resolved:
        return ""
    n_res = norm_name(resolved)
    for code, name in CODE_TO_TEAM.items():
        r = resolve_team_name(name)
        if r and norm_name(r) == n_res:
            return code
    return ""

ratings = {}
for _, r in ratings_df.iterrows():
    team = r["Team"]
    ratings[team] = {
        "attack": float(r["Solver Attack"]),
        "defweak": float(r["Solver Def Weakness"]),
        "defstrength": float(r["Solver Def Strength"]) if "Solver Def Strength" in ratings_df.columns and not pd.isna(r.get("Solver Def Strength")) else (1.0 / float(r["Solver Def Weakness"]) if float(r["Solver Def Weakness"]) else np.nan),
        "attack_index": float(r["Solver Attack Index"]) if "Solver Attack Index" in ratings_df.columns and not pd.isna(r.get("Solver Attack Index")) else np.nan,
        "defweak_index": float(r["Solver DefWeak Index"]) if "Solver DefWeak Index" in ratings_df.columns and not pd.isna(r.get("Solver DefWeak Index")) else np.nan,
        "defstrength_index": float(r["Solver DefStrength Index"]) if "Solver DefStrength Index" in ratings_df.columns and not pd.isna(r.get("Solver DefStrength Index")) else np.nan,
    }

# Ratings output med team code
ratings_out_df = ratings_df.copy()
ratings_out_df.insert(0, "Team Code", ratings_out_df["Team"].apply(code_from_team_name))
ratings_out_df["mu used"] = mu
ratings_out_df["gamma used"] = gamma
ratings_out_df["Solver success"] = solver_success
ratings_out_df["xG MAE"] = solver_xg_mae
ratings_out_df["xG RMSE"] = solver_xg_rmse
ratings_out_df["CS MAE pct points"] = solver_cs_mae

# -----------------------
# Poisson helper
# -----------------------

def poisson_probs(lambda_home, lambda_away, max_goals=12):
    """Returnerer home/draw/away sannsynligheter i prosent fra to xG-lambdaer."""
    ph = [math.exp(-lambda_home) * (lambda_home ** k) / math.factorial(k) for k in range(max_goals + 1)]
    pa = [math.exp(-lambda_away) * (lambda_away ** k) / math.factorial(k) for k in range(max_goals + 1)]
    home = 0.0
    draw = 0.0
    away = 0.0
    for i, p_i in enumerate(ph):
        for j, p_j in enumerate(pa):
            prob = p_i * p_j
            if i > j:
                home += prob
            elif i == j:
                draw += prob
            else:
                away += prob
    total = home + draw + away
    if total > 0:
        home, draw, away = home / total, draw / total, away / total
    return home * 100.0, draw * 100.0, away * 100.0


def projected_values(team_name, opp_name, ha):
    team_attack = ratings[team_name]["attack"]
    team_defweak = ratings[team_name]["defweak"]
    opp_attack = ratings[opp_name]["attack"]
    opp_defweak = ratings[opp_name]["defweak"]

    if ha == "H":
        hfac_team = gamma
        hfac_opp = 1.0 / gamma
    else:
        hfac_team = 1.0 / gamma
        hfac_opp = gamma

    team_xg = mu * team_attack * opp_defweak * hfac_team
    opp_xg = mu * opp_attack * team_defweak * hfac_opp
    cs_pct = math.exp(-opp_xg) * 100.0
    total_xg = team_xg + opp_xg

    if ha == "H":
        win_pct, draw_pct, loss_pct = poisson_probs(team_xg, opp_xg)
    else:
        opp_home_win, draw_pct, team_away_win = poisson_probs(opp_xg, team_xg)
        win_pct = team_away_win
        loss_pct = opp_home_win

    return {
        "Engine xG": team_xg,
        "Engine Opp xG": opp_xg,
        "Engine CS%": cs_pct,
        "Engine Win%": win_pct,
        "Engine Draw%": draw_pct,
        "Engine Loss%": loss_pct,
        "Engine Total xG": total_xg,
    }

# -----------------------
# 1) Les ekte marked fra Market_xG_All_Fixtures_v1
# -----------------------

engine_rows = []
missing_rows = []
market_keys = set()

try:
    market_df = read_sheet(MARKET_XG_SHEET)
except Exception:
    market_df = pd.DataFrame()

market_required = ["Team", "Opponent", "H/A", "Team xG", "Opp xG", "CS%"]
if not market_df.empty and all(c in market_df.columns for c in market_required):
    for _, row in market_df.iterrows():
        team_raw = str(row.get("Team", "")).strip()
        opp_raw = str(row.get("Opponent", "")).strip()
        ha = str(row.get("H/A", "")).strip().upper()
        if not team_raw or not opp_raw or ha not in {"H", "A"}:
            continue
        team_name = resolve_team_name(team_raw)
        opp_name = resolve_team_name(opp_raw)
        if not team_name or not opp_name:
            missing_rows.append({
                "GW": REAL_MARKET_GW,
                "Team Code": code_from_team_name(team_raw),
                "Team": team_raw,
                "Opponent Code": code_from_team_name(opp_raw),
                "Opponent": opp_raw,
                "Issue": "Market_xG-lag eller motstander finnes ikke i Solver-ratingene",
                "Raw Cell": f"{team_raw} vs {opp_raw}",
            })
            continue

        team_code = code_from_team_name(team_name)
        opp_code = code_from_team_name(opp_name)
        market_xg = parse_float(row.get("Team xG"))
        market_opp_xg = parse_float(row.get("Opp xG"))
        market_cs = parse_float(row.get("CS%"))
        market_win = parse_float(row.get("Win%"))
        market_draw = parse_float(row.get("Draw%"))
        market_loss = parse_float(row.get("Loss%"))
        market_total_xg = parse_float(row.get("Total xG"), default=market_xg + market_opp_xg if not pd.isna(market_xg) and not pd.isna(market_opp_xg) else np.nan)

        proj = projected_values(team_name, opp_name, ha)
        xg_diff = proj["Engine xG"] - market_xg if not pd.isna(market_xg) else np.nan
        cs_diff = proj["Engine CS%"] - market_cs if not pd.isna(market_cs) else np.nan

        engine_rows.append({
            "GW": REAL_MARKET_GW,
            "Source": "MARKET",
            "Team Code": team_code,
            "Team": team_name,
            "Opponent Code": opp_code,
            "Opponent": opp_name,
            "H/A": ha,
            "FDR": "",
            "xG": market_xg,
            "Opp xG": market_opp_xg,
            "CS%": market_cs,
            "Win%": market_win,
            "Draw%": market_draw,
            "Loss%": market_loss,
            "Total xG": market_total_xg,
            "Projected xG check": proj["Engine xG"],
            "Projected CS% check": proj["Engine CS%"],
            "xG check diff": xg_diff,
            "CS check diff": cs_diff,
            "Team Attack": ratings[team_name]["attack"],
            "Team DefWeak": ratings[team_name]["defweak"],
            "Opp Attack": ratings[opp_name]["attack"],
            "Opp DefWeak": ratings[opp_name]["defweak"],
            "mu": mu,
            "gamma": gamma,
            "FDR Raw": "",
        })
        market_keys.add((REAL_MARKET_GW, team_code))
else:
    missing_rows.append({
        "GW": REAL_MARKET_GW,
        "Issue": f"{MARKET_XG_SHEET} mangler eller har ikke nødvendige kolonner",
        "Raw Cell": ", ".join(market_required),
    })

# -----------------------
# 2) Les FDR_Overview og beregn projeksjoner
# -----------------------

fdr_df = read_sheet(FDR_SHEET)
if fdr_df.empty:
    raise ValueError(f"{FDR_SHEET} er tomt")
team_col = "Team" if "Team" in fdr_df.columns else fdr_df.columns[0]
missing_gw = [gw for gw in GW_COLUMNS if gw not in fdr_df.columns]
if missing_gw:
    raise ValueError(f"{FDR_SHEET} mangler GW-kolonner: {missing_gw}")


def parse_fdr_cell(value):
    """
    Leser FDR-celle som 'mun 4', 'LEE 2', 'MCI 4'.
    Regel: STORE bokstaver = hjemmekamp, små bokstaver = bortekamp.
    """
    raw = str(value or "").strip()
    if raw == "":
        return None, None, np.nan
    if raw.lower() in {"blank", "bgw", "-", "x"}:
        return None, None, np.nan
    cleaned = raw.replace("(", " ").replace(")", " ").strip()
    m = re.search(r"([A-Za-z]{2,4})\s*([1-5])?", cleaned)
    if not m:
        return None, None, np.nan
    opp_token = m.group(1)
    fdr_num = parse_float(m.group(2), default=np.nan)
    ha = "H" if opp_token.isupper() else "A"
    opp_code = opp_token.upper()
    return opp_code, ha, fdr_num

for _, row in fdr_df.iterrows():
    team_code = str(row.get(team_col, "")).strip().upper()
    if not team_code or team_code == "TEAM":
        continue
    team_name = resolve_team_from_code(team_code)
    if not team_name:
        missing_rows.append({
            "GW": "ALL",
            "Team Code": team_code,
            "Issue": "Team code finnes ikke i Solver-ratingene",
            "Raw Cell": "",
        })
        continue

    for gw in GW_COLUMNS:
        # Ekte marked for GW1 overstyrer FDR-basert projeksjon.
        if (gw, team_code) in market_keys:
            continue

        raw_cell = row.get(gw, "")
        opp_code, ha, fdr_num = parse_fdr_cell(raw_cell)
        if not opp_code:
            continue
        opp_name = resolve_team_from_code(opp_code)
        if not opp_name:
            missing_rows.append({
                "GW": gw,
                "Team Code": team_code,
                "Team": team_name,
                "Opponent Code": opp_code,
                "Issue": "Opponent code finnes ikke i Solver-ratingene",
                "Raw Cell": raw_cell,
            })
            continue

        proj = projected_values(team_name, opp_name, ha)
        source = "PROJECTED" if gw != REAL_MARKET_GW else "PROJECTED_FDR_GW1_FALLBACK"

        engine_rows.append({
            "GW": gw,
            "Source": source,
            "Team Code": team_code,
            "Team": team_name,
            "Opponent Code": opp_code,
            "Opponent": opp_name,
            "H/A": ha,
            "FDR": fdr_num,
            "xG": proj["Engine xG"],
            "Opp xG": proj["Engine Opp xG"],
            "CS%": proj["Engine CS%"],
            "Win%": proj["Engine Win%"],
            "Draw%": proj["Engine Draw%"],
            "Loss%": proj["Engine Loss%"],
            "Total xG": proj["Engine Total xG"],
            "Projected xG check": "",
            "Projected CS% check": "",
            "xG check diff": "",
            "CS check diff": "",
            "Team Attack": ratings[team_name]["attack"],
            "Team DefWeak": ratings[team_name]["defweak"],
            "Opp Attack": ratings[opp_name]["attack"],
            "Opp DefWeak": ratings[opp_name]["defweak"],
            "mu": mu,
            "gamma": gamma,
            "FDR Raw": raw_cell,
        })

engine_df = pd.DataFrame(engine_rows)
missing_df = pd.DataFrame(missing_rows)

if not engine_df.empty:
    engine_df["GW_num"] = engine_df["GW"].str.extract(r"(\d+)").astype(int)
    engine_df = engine_df.sort_values(["GW_num", "Team Code", "Source"]).drop(columns=["GW_num"]).reset_index(drop=True)

# -----------------------
# Wide motorark
# -----------------------

wide_rows = []
if not engine_df.empty:
    for team_code, g in engine_df.groupby("Team Code", sort=True):
        first = g.iloc[0]
        team_name = first["Team"]
        out = {
            "Team Code": team_code,
            "Team": team_name,
            "Solver Attack": ratings[team_name]["attack"],
            "Solver DefWeak": ratings[team_name]["defweak"],
            "Solver DefStrength": ratings[team_name]["defstrength"],
            "Attack Index": ratings[team_name]["attack_index"],
            "DefWeak Index": ratings[team_name]["defweak_index"],
            "DefStrength Index": ratings[team_name]["defstrength_index"],
            "mu": mu,
            "gamma": gamma,
            "Solver Success": solver_success,
            "xG MAE": solver_xg_mae,
            "CS MAE pct": solver_cs_mae,
        }
        for gw in GW_COLUMNS:
            gg = g[g["GW"].eq(gw)]
            if gg.empty:
                out[f"{gw} Source"] = ""
                out[f"{gw} Opp"] = ""
                out[f"{gw} H/A"] = ""
                out[f"{gw} xG"] = ""
                out[f"{gw} Opp xG"] = ""
                out[f"{gw} CS%"] = ""
                out[f"{gw} Win%"] = ""
                out[f"{gw} Draw%"] = ""
                out[f"{gw} Loss%"] = ""
                out[f"{gw} Total xG"] = ""
            else:
                # Hvis både MARKET og fallback finnes, prioriter MARKET.
                if "MARKET" in set(gg["Source"]):
                    r = gg[gg["Source"].eq("MARKET")].iloc[0]
                else:
                    r = gg.iloc[0]
                out[f"{gw} Source"] = r["Source"]
                out[f"{gw} Opp"] = f'{r["Opponent Code"]} ({r["H/A"]})'
                out[f"{gw} H/A"] = r["H/A"]
                out[f"{gw} xG"] = r["xG"]
                out[f"{gw} Opp xG"] = r["Opp xG"]
                out[f"{gw} CS%"] = r["CS%"]
                out[f"{gw} Win%"] = r["Win%"]
                out[f"{gw} Draw%"] = r["Draw%"]
                out[f"{gw} Loss%"] = r["Loss%"]
                out[f"{gw} Total xG"] = r["Total xG"]

        nums = g.copy()
        for c in ["xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"]:
            nums[c] = pd.to_numeric(nums[c], errors="coerce")

        out["Projected GWs Count"] = nums["GW"].nunique()
        out["Market Source Count"] = int((nums["Source"] == "MARKET").sum())
        out["Projection Source Count"] = int(nums["Source"].astype(str).str.startswith("PROJECTED").sum())
        out["Sum xG GW1-5"] = nums["xG"].sum()
        out["Avg xG GW1-5"] = nums["xG"].mean()
        out["Sum Opp xG GW1-5"] = nums["Opp xG"].sum()
        out["Avg Opp xG GW1-5"] = nums["Opp xG"].mean()
        out["Exp CS GW1-5"] = (nums["CS%"] / 100.0).sum()
        out["Avg CS% GW1-5"] = nums["CS%"].mean()
        out["Avg Win% GW1-5"] = nums["Win%"].mean()
        out["Avg Draw% GW1-5"] = nums["Draw%"].mean()
        out["Avg Loss% GW1-5"] = nums["Loss%"].mean()
        out["Best xG GW"] = nums.loc[nums["xG"].idxmax(), "GW"] if nums["xG"].notna().any() else ""
        out["Best xG"] = nums["xG"].max()
        out["Best CS GW"] = nums.loc[nums["CS%"].idxmax(), "GW"] if nums["CS%"].notna().any() else ""
        out["Best CS%"] = nums["CS%"].max()
        out["Projection OK"] = "OK" if nums["GW"].nunique() >= 1 else "MISSING"
        wide_rows.append(out)

wide_df = pd.DataFrame(wide_rows)

# -----------------------
# Summary / kontroll
# -----------------------

expected_solver_teams = len(ratings_df)
market_rows = int((engine_df["Source"] == "MARKET").sum()) if not engine_df.empty else 0
projected_rows = int(engine_df["Source"].astype(str).str.startswith("PROJECTED").sum()) if not engine_df.empty else 0
market_teams = engine_df.loc[engine_df["Source"].eq("MARKET"), "Team Code"].nunique() if not engine_df.empty else 0
projected_teams = engine_df["Team Code"].nunique() if not engine_df.empty else 0
missing_count = len(missing_df)

# Kontrollsum for ekte market vs Solver-rekonstruksjon.
if not engine_df.empty and market_rows > 0:
    mkt = engine_df[engine_df["Source"].eq("MARKET")].copy()
    mkt["abs xG check diff"] = pd.to_numeric(mkt["xG check diff"], errors="coerce").abs()
    mkt["abs CS check diff"] = pd.to_numeric(mkt["CS check diff"], errors="coerce").abs()
    market_xg_check_mae = mkt["abs xG check diff"].mean()
    market_cs_check_mae = mkt["abs CS check diff"].mean()
else:
    market_xg_check_mae = np.nan
    market_cs_check_mae = np.nan

summary_rows = [
    {"Section": "Status", "Metric": "Engine run timestamp", "Value": datetime.now().strftime("%Y-%m-%d %H:%M:%S")},
    {"Section": "Status", "Metric": "Engine OK", "Value": "YES" if solver_success.upper() == "TRUE" and not engine_df.empty else "CHECK"},
    {"Section": "Inputs", "Metric": "Market xG sheet", "Value": MARKET_XG_SHEET},
    {"Section": "Inputs", "Metric": "FDR sheet", "Value": FDR_SHEET},
    {"Section": "Inputs", "Metric": "Solver ratings sheet", "Value": SOLVER_RATINGS_SHEET},
    {"Section": "Inputs", "Metric": "Solver summary sheet", "Value": SOLVER_SUMMARY_SHEET},
    {"Section": "Parameters", "Metric": "mu / league avg xG", "Value": mu},
    {"Section": "Parameters", "Metric": "gamma / home advantage", "Value": gamma},
    {"Section": "Solver", "Metric": "Solver success", "Value": solver_success},
    {"Section": "Solver", "Metric": "xG MAE", "Value": solver_xg_mae},
    {"Section": "Solver", "Metric": "xG RMSE", "Value": solver_xg_rmse},
    {"Section": "Solver", "Metric": "CS MAE pct points", "Value": solver_cs_mae},
    {"Section": "Solver", "Metric": "Regularization", "Value": regularization},
    {"Section": "Solver", "Metric": "Anchor strength", "Value": anchor_strength},
    {"Section": "Rows", "Metric": "Solver teams", "Value": expected_solver_teams},
    {"Section": "Rows", "Metric": "Engine long rows", "Value": len(engine_df)},
    {"Section": "Rows", "Metric": "Market rows", "Value": market_rows},
    {"Section": "Rows", "Metric": "Projected rows", "Value": projected_rows},
    {"Section": "Rows", "Metric": "Teams in engine", "Value": projected_teams},
    {"Section": "Rows", "Metric": "Teams with market source", "Value": market_teams},
    {"Section": "Rows", "Metric": "Missing rows", "Value": missing_count},
    {"Section": "Controls", "Metric": "Market xG check MAE", "Value": market_xg_check_mae},
    {"Section": "Controls", "Metric": "Market CS check MAE pct points", "Value": market_cs_check_mae},
    {"Section": "Controls", "Metric": "Missing note", "Value": "Forventet ved sesongskifte hvis FDR har gamle lag" if missing_count else "Ingen manglende lag"},
]
summary_out_df = pd.DataFrame(summary_rows)


# -----------------------
# Dashboard / instrumentpanel
# -----------------------

def status_icon(ok):
    return "✅" if ok else "⚠️"

projection_quality = "OK" if solver_success.upper() == "TRUE" and not engine_df.empty else "CHECK"
missing_note = "Sesongovergang / FDR-lag mangler" if missing_count else "Ingen missing"
source_note = "GW1 = MARKET | GW2-GW5 = PROJECTED"
solver_run_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

dashboard_rows = [
    {"Market Engine v1": "Status", "Value": f"{status_icon(projection_quality == 'OK')} {projection_quality}", "Note": "Projection Quality"},
    {"Market Engine v1": "μ / league avg xG", "Value": mu, "Note": "Brukes i alle projected xG-beregninger"},
    {"Market Engine v1": "γ / home advantage", "Value": gamma, "Note": "Hjemmefaktor; borte bruker 1/γ"},
    {"Market Engine v1": "Solver run", "Value": solver_run_ts, "Note": "Tidspunkt dette engine-scriptet ble kjørt"},
    {"Market Engine v1": "Solver success", "Value": solver_success, "Note": "Fra Market_Solver_Summary_v1"},
    {"Market Engine v1": "Solver xG MAE", "Value": solver_xg_mae, "Note": "Lavere er bedre"},
    {"Market Engine v1": "Solver CS MAE pct points", "Value": solver_cs_mae, "Note": "Lavere er bedre"},
    {"Market Engine v1": "Source", "Value": source_note, "Note": "MARKET = ekte oddsdata; PROJECTED = solvermotor"},
    {"Market Engine v1": "Rows", "Value": f"{len(engine_df)} total | {market_rows} market | {projected_rows} projected", "Note": "Engine_Long"},
    {"Market Engine v1": "Teams", "Value": f"{projected_teams} i engine | {expected_solver_teams} i solver", "Note": "Bør bli 20 når FDR er oppdatert"},
    {"Market Engine v1": "Missing", "Value": missing_count, "Note": missing_note},
    {"Market Engine v1": "Market xG check MAE", "Value": market_xg_check_mae, "Note": "Kontroll mot ekte GW1 market"},
    {"Market Engine v1": "Market CS check MAE", "Value": market_cs_check_mae, "Note": "Kontroll mot ekte GW1 market"},
]
dashboard_df = pd.DataFrame(dashboard_rows)
dashboard_df = round_numeric(dashboard_df, 4)

# -----------------------
# Round numeric cols
# -----------------------

engine_df = round_numeric(engine_df, 4)
wide_df = round_numeric(wide_df, 4)
ratings_out_df = round_numeric(ratings_out_df, 4)
summary_out_df = round_numeric(summary_out_df, 4)

if missing_df.empty:
    missing_df = pd.DataFrame([{"Status": "Ingen manglende lag/motstandere"}])

# -----------------------
# Write sheets
# -----------------------

update_worksheet(ENGINE_DASHBOARD_OUT, dashboard_df, min_rows=40, min_cols=8)
update_worksheet(ENGINE_SUMMARY_OUT, summary_out_df, min_rows=80, min_cols=10)
update_worksheet(ENGINE_RATINGS_OUT, ratings_out_df, min_rows=80, min_cols=25)
update_worksheet(ENGINE_LONG_OUT, engine_df, min_rows=180, min_cols=35)
update_worksheet(ENGINE_WIDE_OUT, wide_df, min_rows=80, min_cols=70)
update_worksheet(ENGINE_MISSING_OUT, missing_df, min_rows=80, min_cols=15)

print("")
print("FERDIG: Jason Market Engine v1")
print(f"Engine long rows: {len(engine_df)}")
print(f"Market rows: {market_rows}")
print(f"Projected rows: {projected_rows}")
print(f"Missing rows: {missing_count}")
print("")
print("Sjekk arkene:")
print(f"- {ENGINE_DASHBOARD_OUT}")
print(f"- {ENGINE_SUMMARY_OUT}")
print(f"- {ENGINE_RATINGS_OUT}")
print(f"- {ENGINE_LONG_OUT}")
print(f"- {ENGINE_WIDE_OUT}")
print(f"- {ENGINE_MISSING_OUT}")

mu: 1.5430
gamma: 1.0865
solver success: TRUE
Oppdaterer Market_Engine_Dashboard_v1...
Oppdaterer Market_Engine_Summary_v1...
Oppdaterer Market_Engine_Ratings_v1...
Oppdaterer Market_Engine_Long_v1...
Oppdaterer Market_Engine_Wide_v1...
Oppdaterer Market_Engine_Missing_v1...

FERDIG: Jason Market Engine v1
Engine long rows: 76
Market rows: 20
Projected rows: 56
Missing rows: 15

Sjekk arkene:
- Market_Engine_Dashboard_v1
- Market_Engine_Summary_v1
- Market_Engine_Ratings_v1
- Market_Engine_Long_v1
- Market_Engine_Wide_v1
- Market_Engine_Missing_v1
